In [2]:
# Notebook de blend — combine les prédictions de plusieurs modèles
# Les fichiers .npz de prédictions sont générés par les notebooks d'entraînement
# et sauvegardés dans /kaggle/working/outputs/ de chaque notebook
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
print('OK')

OK


In [3]:
WORK     = Path('/kaggle/working')
PRED_MIN = 30.0
PRED_MAX = 3600.0

# ------------------------------------------------------------------------
# CHARGEMENT DES PREDICTIONS
#
# Chaque modèle sauvegarde ses prédictions sur le test set dans un CSV.
# On charge les prédictions de chaque modèle et on les aligne par image_id.
# ------------------------------------------------------------------------

model_csvs = {
    'convnextv2_base_mm'    : Path('/kaggle/input/notebooks/yacineabdelouhab/kaggle-convnextv2-base-multimodal/submission_convnextv2_base_mm.csv'),
    'deit3_base_mm'         : Path('/kaggle/input/notebooks/yacineabdelouhab/kaggle-deit3-base-multimodal/submission_deit3_base_mm.csv'),
    'convnextv2_base_augmax': Path('/kaggle/input/notebooks/yacineabdelouhab/kaggle-convnextv2-base-augmax/submission_convnextv2_base_augmax.csv'),
    'swin_base_mm'          : Path('/kaggle/input/notebooks/yacineabdelouhab/kaggle-swin-base-multimodal/submission_swin_base_mm.csv'),
    'coatnet1_imgonly'      : Path('/kaggle/input/notebooks/yacineabdelouhab/kaggle-coatnet1-imgonly/submission_coatnet1_imgonly.csv'),
}

preds_dict = {}
for name, path in model_csvs.items():
    if path.exists():
        df = pd.read_csv(path).sort_values('image_id').reset_index(drop=True)
        preds_dict[name] = df['calories'].values
        print(f'  {name}: {len(df)} preds, mean={df["calories"].mean():.0f}')
    else:
        print(f'  SKIP (not found): {name}')

image_ids = pd.read_csv(list(model_csvs.values())[0]).sort_values('image_id')['image_id'].values
print(f'\n{len(preds_dict)} modèles chargés')

  convnextv2_base_mm: 547 preds, mean=253
  deit3_base_mm: 547 preds, mean=223
  convnextv2_base_augmax: 547 preds, mean=310
  swin_base_mm: 547 preds, mean=301
  coatnet1_imgonly: 547 preds, mean=221

5 modèles chargés


In [4]:
# ------------------------------------------------------------------------
# BLEND PAR MOYENNE PONDEREE
#
# On combine les prédictions de chaque modèle avec des poids.
# Les poids ont été optimisés sur le validation set (étape 1) via
# l'algorithme de Nelder-Mead (optimisation sans gradient).
#
# Principe : minimiser le MAE sur le val set en cherchant les poids
# optimaux w1, w2, ... tels que sum(wi) = 1 et wi >= 0.
# ------------------------------------------------------------------------

models  = list(preds_dict.keys())
M       = len(models)
kag_mat = np.stack([preds_dict[n] for n in models])  # (M, N)

# Poids optimisés depuis l'étape 1 (Nelder-Mead sur val set)
WEIGHTS = {
    'convnextv2_base_mm'    : 0.55,
    'deit3_base_mm'         : 0.35,
    'convnextv2_base_augmax': 0.10,
}

w = np.array([WEIGHTS.get(n, 1.0/M) for n in models])
w = np.abs(w); w /= w.sum()

blend = (kag_mat * w[:, None]).sum(axis=0)
blend = np.clip(blend, PRED_MIN, PRED_MAX)

for n, wi in zip(models, w):
    print(f'  {n}: {wi:.4f}')
print(f'\nBlend: mean={blend.mean():.0f} min={blend.min():.0f} max={blend.max():.0f}')

  convnextv2_base_mm: 0.3929
  deit3_base_mm: 0.2500
  convnextv2_base_augmax: 0.0714
  swin_base_mm: 0.1429
  coatnet1_imgonly: 0.1429

Blend: mean=252 min=63 max=1461


In [5]:
sub = pd.DataFrame({'image_id': image_ids, 'calories': blend.round(2)})
sub_path = WORK / 'submission_blend.csv'
sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
print(sub.head())

Saved: /kaggle/working/submission_blend.csv
    image_id  calories
0  test_0000     83.23
1  test_0001    152.80
2  test_0002    147.72
3  test_0003    211.64
4  test_0004    162.70
